# Task 2.2.5: Configuring and Understanding the YOLOv5 Model
## Configuring the configuration files of YOLOv5 and understanding YOLOv5's architecture

In [8]:
# Navigate to the models directory of YOLOv5 repository
%cd "/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/yolo/yolov5/models"
# You can view and edit the configuration files here using any text editor or even directly in the notebook.
# If you are unable to edit the file in the notebook, find it in its directory and open it separately.
# Display the content of the configuration file for the small version of YOLOv5
!cat yolov5s.yaml

/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/yolo/yolov5/models
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Parameters
nc: 80 # number of classes
depth_multiple: 0.33 # model depth multiple
width_multiple: 0.50 # layer channel multiple
anchors:
  - [10, 13, 16, 30, 33, 23] # P3/8
  - [30, 61, 62, 45, 59, 119] # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2
    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]], # 9
  ]

# YOLOv5 v6.0 head
head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]], # cat backbone P4
    [-1, 3, C3, [512,

In [9]:
# We can load the YOLOv5 model and display its architecture directly in the notebook
import torch
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)  # Load the YOLOv5 small model
print(model)  # Display the model architecture

Using cache found in /Users/zignov/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-5-7 Python-3.11.9 torch-2.11.0 CPU

100%|██████████| 14.1M/14.1M [00:00<00:00, 14.8MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


AutoShape(
  (model): DetectMultiBackend(
    (model): DetectionModel(
      (model): Sequential(
        (0): Conv(
          (conv): Conv2d(3, 32, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2))
          (act): SiLU(inplace=True)
        )
        (1): Conv(
          (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (act): SiLU(inplace=True)
        )
        (2): C3(
          (cv1): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv3): Conv(
            (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (m): Sequential(
            (0): Bottleneck(
              (cv1): Conv(
                (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
  

#### Preparing YOLOv5 for Training

In [10]:
# Reading the content of the YOLOv5s.yaml configuration file
with open('/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/yolo/yolov5/models/yolov5s.yaml', 'r') as file:
    content = file.readlines()

# Displaying the content
for line in content:
    print(line, end='')

# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Parameters
nc: 80 # number of classes
depth_multiple: 0.33 # model depth multiple
width_multiple: 0.50 # layer channel multiple
anchors:
  - [10, 13, 16, 30, 33, 23] # P3/8
  - [30, 61, 62, 45, 59, 119] # P4/16
  - [116, 90, 156, 198, 373, 326] # P5/32

# YOLOv5 v6.0 backbone
backbone:
  # [from, number, module, args]
  [
    [-1, 1, Conv, [64, 6, 2, 2]], # 0-P1/2
    [-1, 1, Conv, [128, 3, 2]], # 1-P2/4
    [-1, 3, C3, [128]],
    [-1, 1, Conv, [256, 3, 2]], # 3-P3/8
    [-1, 6, C3, [256]],
    [-1, 1, Conv, [512, 3, 2]], # 5-P4/16
    [-1, 9, C3, [512]],
    [-1, 1, Conv, [1024, 3, 2]], # 7-P5/32
    [-1, 3, C3, [1024]],
    [-1, 1, SPPF, [1024, 5]], # 9
  ]

# YOLOv5 v6.0 head
head: [
    [-1, 1, Conv, [512, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, "nearest"]],
    [[-1, 6], 1, Concat, [1]], # cat backbone P4
    [-1, 3, C3, [512, False]], # 13

    [-1, 1, Conv, [256, 1, 1]],
    [-1, 1, nn.Upsample, [None, 2, 

In [11]:
# Modifying the yolov5s.yaml file directly using Python, we will save it as a new file.
# Modifying the number of classes (nc) in the configuration
new_content = []
for line in content:
    if 'nc:' in line:
        line = 'nc: 1  # number of classes\n'
    new_content.append(line)

# Saving the modified content back to the file (saving as a new file)
with open('/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/yolo/yolov5/models/custom_yolov5s.yaml', 'w') as file:
    file.writelines(new_content)

In [ ]:
# If we choose to customize these anchors, it’s best to use the automatic anchor calculation provided by YOLOv5.
# This can be done by running the following command in the terminal:
'''
python3 train.py --img 640 --batch 16 --epochs 100 --data my_dataset.yaml --cfg yolov5s.yaml --weights yolov5s.pt --device 0 --name yolov5s_results --cache --evolve
'''

TODO: Create a new YAML file, and name it `my_dataset.yaml`.

TODO: Open the file in a text editor and add the following contents:

```
# path to your training images and labels
train: C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_training

# path to your validation images and labels
val: C:/Users/vasja/Desktop/AIM4VET/LM2TU2/cardata/my_validation

# number of classes
nc: 1  

# class names
names: ['car']  
```